In [ ]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("feature_files/ZooMus_115_features_raw.csv")
df.head()

,times,looking_binary,rmsVals,brightVals,roughVals,fluxVals,pitchVals
0,0.0,1,0.000207,NaN,0.000000,0.000000,NaN
1,0.5,1,0.019272,1920.233394,0.454915,73.301043,NaN
2,1.0,1,0.030926,1600.109227,7736.373216,372.690470,225.121755
3,1.5,0,0.044479,1387.097824,74598.019444,579.206167,737.526028
4,2.0,0,0.043118,1984.941618,13979.747434,517.882897,316.525320


In [3]:
# create previous looking column
# shift(1) gives us previous row values
# Row 5’s prev_looking now equals row 4’s looking_binary.
df["prev_looking"] = df["looking_binary"].shift(1)
df.head()

,times,looking_binary,rmsVals,brightVals,roughVals,fluxVals,pitchVals,prev_looking
0,0.0,1,0.000207,NaN,0.000000,0.000000,NaN,NaN
1,0.5,1,0.019272,1920.233394,0.454915,73.301043,NaN,1.0
2,1.0,1,0.030926,1600.109227,7736.373216,372.690470,225.121755,1.0
3,1.5,0,0.044479,1387.097824,74598.019444,579.206167,737.526028,1.0
4,2.0,0,0.043118,1984.941618,13979.747434,517.882897,316.525320,0.0


In [4]:
# Detect switches
df["switch"] = df["looking_binary"] - df["prev_looking"]
df.head()

,times,looking_binary,rmsVals,brightVals,roughVals,fluxVals,pitchVals,prev_looking,switch
0,0.0,1,0.000207,NaN,0.000000,0.000000,NaN,NaN,NaN
1,0.5,1,0.019272,1920.233394,0.454915,73.301043,NaN,1.0,0.0
2,1.0,1,0.030926,1600.109227,7736.373216,372.690470,225.121755,1.0,0.0
3,1.5,0,0.044479,1387.097824,74598.019444,579.206167,737.526028,1.0,-1.0
4,2.0,0,0.043118,1984.941618,13979.747434,517.882897,316.525320,0.0,0.0


In [5]:
# Clean first row
df['switch'] = df['switch'].fillna(0)

In [6]:
# Sanity Check
df["switch"].value_counts()

switch
 0.0    337
-1.0     17
 1.0     17
Name: count, dtype: int64

In [8]:
window_size = 10  # 5 seconds (10 bins of 0.5 sec)
switch_rows = df[df["switch"] != 0].copy()
switch_rows

,times,looking_binary,rmsVals,brightVals,roughVals,fluxVals,pitchVals,prev_looking,switch
3,1.5,0,0.044479,1387.097824,74598.019444,579.206167,737.526028,1.0,-1.0
8,4.0,1,0.022931,2791.624940,1202.295862,296.518631,NaN,0.0,1.0
18,9.0,0,0.029046,4571.528213,3676.218122,254.375980,234.309409,1.0,-1.0
21,10.5,1,0.033047,5671.408775,13177.985175,321.297149,321.689121,0.0,1.0
39,19.5,0,0.038847,2335.056348,9263.577353,544.866508,350.255268,1.0,-1.0
43,21.5,1,0.037404,2639.788993,39061.470264,225.930592,380.410685,0.0,1.0
64,32.0,0,0.048954,1997.125661,12063.083501,691.484944,319.762366,1.0,-1.0
67,33.5,1,0.041474,4294.244544,11320.855298,614.856921,399.780463,0.0,1.0
69,34.5,0,0.043381,2338.913837,12097.724454,414.051448,98.383932,1.0,-1.0
71,35.5,1,0.058362,2122.168123,53381.503855,853.688600,326.126754,0.0,1.0


In [11]:
event_data = []
# Lopp through switches
for idx in switch_rows.index:
    # Only compute if enough bins before switch
    if idx >= window_size:
        # selectes the rows before the switch excluding the switch
        window = df.loc[idx-window_size: idx-1]

        # averaging each feature
        mean_rms = window["rmsVals"].mean()
        mean_bright = window["brightVals"].mean()
        mean_rough = window["roughVals"].mean()
        mean_flux = window["fluxVals"].mean()
        mean_pitch = window["pitchVals"].mean()

        # 1 = switch to looking, -1= switch away
        switch_type = df.loc[idx, "switch"]

        event_data.append({
            "switch_type": switch_type,
            "mean_rms": mean_rms,
            "mean_brightness": mean_bright,
            "mean_roughness": mean_rough,
            "mean_flux": mean_flux,
            "mean_pitch": mean_pitch
        })

# convert to df
event_df = pd.DataFrame(event_data)
event_df.head()


,switch_type,mean_rms,mean_brightness,mean_roughness,mean_flux,mean_pitch
0,-1.0,0.031476,3920.334310,10418.338959,413.095396,439.204085
1,1.0,0.032137,3517.354141,12207.108024,385.920007,423.165533
2,-1.0,0.032798,2754.268709,21882.129131,435.250800,553.433326
3,1.0,0.043172,1992.843575,31103.798141,458.835729,506.054150
4,-1.0,0.053090,3630.428778,32169.975789,558.237814,360.119249


In [15]:
# splitting into two groups
to_looking = event_df[event_df["switch_type"] == 1]
away = event_df[event_df["switch_type"] == -1]

print("To looking:", len(to_looking))
print("Away:", len(away))


To looking: 16
Away: 16


In [ ]:
# Compute means
to_means = to_looking.mean(numeric_only=True)
away_means = away.mean(numeric_only=True)

print("To looking means:\n", to_means)
print("\nAway means:\n", away_means)

To looking means:
 switch_type            1.000000
mean_rms               0.041740
mean_brightness     3146.242215
mean_roughness     25060.520694
mean_flux            470.789929
mean_pitch           346.495548
dtype: float64

Away means:
 switch_type           -1.000000
mean_rms               0.041171
mean_brightness     3316.209403
mean_roughness     21982.301968
mean_flux            490.219734
mean_pitch           376.052832
dtype: float64


In [22]:
# Compute 85% CIs
def mean_ci(series):
    series = series.dropna()  
    
    mean = np.mean(series)
    std = np.std(series, ddof=1)
    n = len(series)
    se = std / np.sqrt(n)
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    return mean, ci_low, ci_high, n

In [23]:
# Apply to each feature

def print_ci(feature_name, group1, group2):
    
    mean1, low1, high1, n1 = mean_ci(group1)
    mean2, low2, high2, n2 = mean_ci(group2)
    
    print(f"\n===== {feature_name} =====")
    
    print(f"Switch TO looking (n = {n1}):")
    print(f"  Mean = {mean1:.4f}")
    print(f"  95% CI = [{low1:.4f}, {high1:.4f}]")
    
    print(f"\nSwitch AWAY (n = {n2}):")
    print(f"  Mean = {mean2:.4f}")
    print(f"  95% CI = [{low2:.4f}, {high2:.4f}]")

    

In [24]:
print_ci("RMS", to_looking["mean_rms"], away["mean_rms"])
print_ci("Brightness", to_looking["mean_brightness"], away["mean_brightness"])
print_ci("Roughness", to_looking["mean_roughness"], away["mean_roughness"])
print_ci("Flux", to_looking["mean_flux"], away["mean_flux"])
print_ci("Pitch", to_looking["mean_pitch"], away["mean_pitch"])


===== RMS =====
Switch TO looking (n = 16):
  Mean = 0.0417
  95% CI = [0.0348, 0.0486]

Switch AWAY (n = 16):
  Mean = 0.0412
  95% CI = [0.0356, 0.0467]

===== Brightness =====
Switch TO looking (n = 16):
  Mean = 3146.2422
  95% CI = [2911.5381, 3380.9464]

Switch AWAY (n = 16):
  Mean = 3316.2094
  95% CI = [3136.7239, 3495.6949]

===== Roughness =====
Switch TO looking (n = 16):
  Mean = 25060.5207
  95% CI = [13462.0598, 36658.9816]

Switch AWAY (n = 16):
  Mean = 21982.3020
  95% CI = [16428.3456, 27536.2583]

===== Flux =====
Switch TO looking (n = 16):
  Mean = 470.7899
  95% CI = [398.2171, 543.3628]

Switch AWAY (n = 16):
  Mean = 490.2197
  95% CI = [426.7022, 553.7373]

===== Pitch =====
Switch TO looking (n = 16):
  Mean = 346.4955
  95% CI = [316.6834, 376.3077]

Switch AWAY (n = 16):
  Mean = 376.0528
  95% CI = [347.2375, 404.8681]


In [ ]:
# Brightness
# Away seems slightly brighter on average. Maybe a slight tendency for higher brightness before switching away

# Roughness
# huge intervals. This suggests roughness is unstable and noisy in this dataset?

# Pitch
# Away events seem preceded by slightly higher pitch.
